## Reinício do ambiente de streaming (opcional)

Esta célula reinicia as duas streams estruturadas (`bus_telemetry_raw` e `charger_telemetry_raw`) do zero: pára as streams ativas, remove as tabelas Delta e as respetivas views, e limpa os checkpoints associados. É útil depois de uma alteração ao schema de ingestão ou de uma sessão de cluster terminada de forma anómala, mas não deve ser corrida em utilização normal — reiniciar implica perder a telemetria já recolhida. A célula seguinte confirma que a tabela `bus_telemetry` deixou de existir, validando que a limpeza foi bem-sucedida.

In [0]:
for stream in spark.streams.active:
    print(f"A parar stream: {stream.id}")
    stream.stop()

import time
time.sleep(5)

spark.sql("DROP TABLE IF EXISTS bus_telemetry_raw")
spark.sql("DROP TABLE IF EXISTS charger_telemetry_raw")
spark.sql("DROP VIEW IF EXISTS bus_telemetry")
spark.sql("DROP VIEW IF EXISTS charger_telemetry")
print("Tabelas e views eliminadas")

dbutils.fs.rm("dbfs:/Workspace/checkpoints/bus_telemetry", True)
dbutils.fs.rm("dbfs:/Workspace/checkpoints/charger_telemetry", True)
print("Checkpoints eliminados")

print("\nRESET COMPLETO! Agora executa as celulas de stream novamente")

Tabelas e views eliminadas
Checkpoints eliminados

RESET COMPLETO! Agora executa as celulas de stream novamente


In [0]:
try:
    spark.sql("SELECT COUNT(*) FROM bus_telemetry").show()
    print("tabela ainda existe")
except:
    print("tabela eliminada")

tabela eliminada


# STCP - Análise de Cenário GTFS Real

Análise de autocarros elétricos em rotas reais do Porto:
- BUS-001: Rota 200 (Bolhão - Castelo Queijo)
- BUS-002: Rota 201 (Aliados - Viso)
- BUS-003: Rota 202 (Aliados - Passeio Alegre)

Última atualização: Outubro 2026

## Passo 1: Configuração Event Hub

Ligação ao Azure Event Hub para receber telemetria em tempo real.

In [0]:
# Configuração do Event Hub
# Estes parâmetros estabelecem a ligação ao Azure Event Hub

# Nome do namespace onde está o Event Hub
eh_namespace = "ehns-stcp-bus"

# Nome do Event Hub específico que contém a telemetria
eh_name = "stcp-telemetry"

# Nome da política de acesso (Manage permite ler e escrever)
eh_key_name = "RootManageSharedAccessKey"

# Chave de acesso (Primary Key da política Manage)
eh_key = "ZdvXUobLzuB4r4I+9X1qBVOkIHLZ62gIw+AEhPRpFiY="

# String de autenticação SASL para Kafka
# Formato específico para Event Hub: usa a connection string como password
eh_sasl = f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username="$ConnectionString" password="Endpoint=sb://{eh_namespace}.servicebus.windows.net/;SharedAccessKeyName={eh_key_name};SharedAccessKey={eh_key};EntityPath={eh_name}";'

print("Configuração concluída com sucesso")

Configuração concluída com sucesso


## Passo 3: Ler Dados do Event Hub

Criar streaming DataFrame que lê continuamente do Event Hub.

**Configuração**:
- `startingOffsets: "latest"` - Apenas dados NOVOS
- `startingOffsets: "earliest"` - TODO o histórico + novos

Configurado para **"latest"**.

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

# Criar streaming DataFrame
df_stream = (spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", f"{eh_namespace}.servicebus.windows.net:9093")
    .option("subscribe", eh_name)
    .option("kafka.sasl.mechanism", "PLAIN")
    .option("kafka.security.protocol", "SASL_SSL")
    .option("kafka.sasl.jaas.config", eh_sasl)
    .option("kafka.request.timeout.ms", "60000")
    .option("kafka.session.timeout.ms", "30000")
    .option("failOnDataLoss", "false")
    .option("startingOffsets", "latest")
    .load()
)

print("Stream configurado!")
print("Colunas disponiveis:", df_stream.columns)

Stream configurado!
Colunas disponiveis: ['key', 'value', 'topic', 'partition', 'offset', 'timestamp', 'timestampType']


## Passo 4: Parse do JSON e Separacao de Dados
**Parse JSON** = Transformar esse texto em **colunas**:
Processar JSON e separar em duas streams:
- Autocarros: batteryKwh, batteryLevel, batteryTemperature, latitude, longitude, state
- Estacoes: chargerTemperature, currentEfficiency, currentPower, energyDelivered, state

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

# Schema COMPLETO do JSON (com telemetry wrapper)
full_schema = StructType([
    StructField("deviceId", StringType(), True),
    StructField("telemetry", StructType([
        StructField("batteryKwh", DoubleType(), True),
        StructField("batteryLevel", DoubleType(), True),
        StructField("batteryTemperature", DoubleType(), True),
        StructField("latitude", DoubleType(), True),
        StructField("longitude", DoubleType(), True),
        StructField("state", StringType(), True),
        StructField("instantConsumption", DoubleType(), True),
        StructField("remainingRange", DoubleType(), True),
        StructField("distanceTraveled", DoubleType(), True),
        StructField("tripCount", IntegerType(), True),
        StructField("currentStop", StringType(), True),
        StructField("nextStop", StringType(), True),
        StructField("chargerTemperature", DoubleType(), True),
        StructField("currentEfficiency", DoubleType(), True),
        StructField("currentPower", DoubleType(), True),
        StructField("energyDelivered", DoubleType(), True),
        StructField("connectedBus", StringType(), True),
        StructField("totalEnergyDelivered", DoubleType(), True)
    ]), True)
])

# Parse JSON completo
df_parsed = df_stream.select(
    from_json(col("value").cast("string"), full_schema).alias("data"),
    col("timestamp").alias("enqueued_time")
).withColumn("event_time", current_timestamp())

# Extrair dados dos autocarros
df_bus_stream = df_parsed.filter(col("data.deviceId").rlike("BUS-")).select(
    col("event_time"),
    col("data.deviceId").alias("device_id"),
    col("enqueued_time"),
    col("data.telemetry.batteryLevel").alias("batteryLevel"),
    col("data.telemetry.batteryKwh").alias("batteryKwh"),
    col("data.telemetry.batteryTemperature").alias("batteryTemperature"),
    col("data.telemetry.latitude").alias("latitude"),
    col("data.telemetry.longitude").alias("longitude"),
    col("data.telemetry.state").alias("state"),
    col("data.telemetry.instantConsumption").alias("instantConsumption"),
    col("data.telemetry.remainingRange").alias("remainingRange"),
    col("data.telemetry.distanceTraveled").alias("distanceTraveled"),
    col("data.telemetry.tripCount").alias("tripCount"),
    col("data.telemetry.currentStop").alias("currentStop"),
    col("data.telemetry.nextStop").alias("nextStop")
)

# Extrair dados das estacoes
df_charger_stream = df_parsed.filter(col("data.deviceId").rlike("CS-")).select(
    col("event_time"),
    col("data.deviceId").alias("device_id"),
    col("enqueued_time"),
    col("data.telemetry.chargerTemperature").alias("chargerTemperature"),
    col("data.telemetry.currentEfficiency").alias("currentEfficiency"),
    col("data.telemetry.currentPower").alias("currentPower"),
    col("data.telemetry.energyDelivered").alias("energyDelivered"),
    col("data.telemetry.state").alias("state"),
    col("data.telemetry.connectedBus").alias("connectedBus"),
    col("data.telemetry.totalEnergyDelivered").alias("totalEnergyDelivered"),
    col("data.telemetry.latitude").alias("latitude"),
    col("data.telemetry.longitude").alias("longitude")
)

print("Schema actualizado com todos os campos!")

Schema actualizado com todos os campos!


## Passo 5: Gravar Streams em Delta Tables

Criar tabelas Delta para armazenar telemetria:
- bus_telemetry
- charger_telemetry

In [0]:
import threading
import time

# Inicia as duas streams e mantém-nas vivas: se alguma parar (sessão reciclada,
# cluster instável, etc.), a thread em segundo plano reinicia-a automaticamente.
# Não bloqueia o notebook — "Run All" continua normalmente para as células seguintes.

def _start_streams():
    global query_bus, query_charger
    query_bus = df_bus_stream.writeStream.format("delta").outputMode("append") \
        .option("checkpointLocation", "dbfs:/Workspace/checkpoints/bus_telemetry") \
        .table("bus_telemetry_raw")
    query_charger = df_charger_stream.writeStream.format("delta").outputMode("append") \
        .option("checkpointLocation", "dbfs:/Workspace/checkpoints/charger_telemetry") \
        .table("charger_telemetry_raw")

def _watch_streams():
    while True:
        time.sleep(30)
        if not query_bus.isActive or not query_charger.isActive:
            print(f"[{time.strftime('%H:%M:%S')}] Stream parou — a reiniciar...")
            _start_streams()

_start_streams()
threading.Thread(target=_watch_streams, daemon=True).start()

print("Streams a gravar em Delta Tables, com reinício automático em caso de falha.")
print("Aguarda 30-60 segundos para dados chegarem...")

In [0]:
# Criar views com os nomes originais que filtram apenas dados dos ultimos 2 minutos
# Assim todas as queries continuam a usar bus_telemetry e charger_telemetry
# mas mostram sempre apenas dados recentes, evitando dados antigos do Event Hub

spark.sql("""
    CREATE OR REPLACE VIEW bus_telemetry AS
    SELECT * FROM bus_telemetry_raw
    WHERE event_time >= current_timestamp() - INTERVAL 10 MINUTES
""")

spark.sql("""
    CREATE OR REPLACE VIEW charger_telemetry AS
    SELECT * FROM charger_telemetry_raw
    WHERE event_time >= current_timestamp() - INTERVAL 10 MINUTES
""")

print("Views criadas com filtro de 10 minutos")
print("Todas as queries continuam a usar os mesmos nomes de tabela")

Views criadas com filtro de 10 minutos
Todas as queries continuam a usar os mesmos nomes de tabela


### Confirmação de streams ativos

Verificação rápida de que as duas streams continuam em execução depois do arranque na célula anterior, antes de se avançar para a consulta dos dados recebidos.

In [0]:
# Ver se streams estão ativos
print("=== STREAMS ATIVOS ===")
for stream in spark.streams.active:
    print(f"Stream: {stream.name} | Status: {stream.status}")
print(f"Total: {len(spark.streams.active)} streams")

=== STREAMS ATIVOS ===
Stream: None | Status: {'message': 'Writing offsets to log', 'isDataAvailable': False, 'isTriggerActive': True}
Stream: None | Status: {'message': 'Processing new data', 'isDataAvailable': True, 'isTriggerActive': True}
Total: 2 streams


## Passo 6: Verificar Dados

Aguardar 60 segundos e verificar se dados estão a chegar.

## Visualização de Dados - Autocarros

Todas as características dos autocarros em tempo real.

In [0]:
print("=== AUTOCARROS - TODAS AS CARACTERÍSTICAS ===")

spark.sql("""
    SELECT 
        device_id        AS deviceId,
        batteryLevel     AS batteryLevel,
        batteryKwh       AS batteryKwh,
        batteryTemperature AS batteryTemperature,
        latitude         AS latitude,
        longitude        AS longitude,
        state            AS state,
        instantConsumption AS instantConsumption,
        remainingRange   AS remainingRange,
        distanceTraveled AS distanceTraveled,
        tripCount        AS tripCount,
        currentStop      AS currentStop,
        nextStop         AS nextStop,
        date_format(event_time, 'dd-MM-yyyy HH:mm:ss') AS event_time
    FROM bus_telemetry
    ORDER BY device_id, event_time DESC
    LIMIT 2000
""").show(2000, truncate=False)

=== AUTOCARROS - TODAS AS CARACTERÍSTICAS ===
+--------+------------+----------+------------------+---------+---------+-----------------+------------------+--------------+----------------+---------+-----------------------+-----------------------+-------------------+
|deviceId|batteryLevel|batteryKwh|batteryTemperature|latitude |longitude|state            |instantConsumption|remainingRange|distanceTraveled|tripCount|currentStop            |nextStop               |event_time         |
+--------+------------+----------+------------------+---------+---------+-----------------+------------------+--------------+----------------+---------+-----------------------+-----------------------+-------------------+
|BUS-001 |17.75       |62.12     |23.17             |41.148164|-8.608651|headingToCharging|60.0              |51.77         |8189.84         |12       |Pr. D. João I          |Central de Carregamento|16-10-2026 10:42:58|
|BUS-001 |20.27       |70.96     |23.14             |41.152665|-8.6304

In [0]:
print("=== ESTAÇÕES DE CARREGAMENTO - TODAS AS CARACTERÍSTICAS ===")

spark.sql("""
    SELECT 
        device_id,
        state,
        currentPower,
        currentEfficiency,
        chargerTemperature,
        energyDelivered,
        totalEnergyDelivered,
        connectedBus,
        latitude,
        longitude,
        date_format(event_time, 'dd-MM-yyyy HH:mm:ss') as data_hora
    FROM charger_telemetry
    ORDER BY event_time DESC
    LIMIT 20
""").show(20, truncate=False)

=== ESTAÇÕES DE CARREGAMENTO - TODAS AS CARACTERÍSTICAS ===
+---------+---------+------------+-----------------+------------------+---------------+--------------------+------------+---------+---------+-------------------+
|device_id|state    |currentPower|currentEfficiency|chargerTemperature|energyDelivered|totalEnergyDelivered|connectedBus|latitude |longitude|data_hora          |
+---------+---------+------------+-----------------+------------------+---------------+--------------------+------------+---------+---------+-------------------+
|CS-003   |occupied |120.0       |91.61            |31.43             |16.17          |NULL                |BUS-003     |41.152681|-8.579678|16-10-2026 10:39:14|
|CS-001   |occupied |80.0        |91.39            |45.0              |29.94          |NULL                |BUS-004     |41.152661|-8.579658|16-10-2026 10:39:14|
|CS-002   |fault    |0.0         |0.0              |28.46             |0.0            |NULL                |NULL        |41.152671

## Passo 7: Tabela de Referência de Paragens Reais (route_stops)

86 paragens reais da STCP (30 da Linha 200, 26 da Linha 201, 30 da Linha 202), com nomes, coordenadas e distância acumulada real. Esta tabela é usada pela Notebook 1.1 ("Paragem Mais Próxima" / "Quantas Paragens Faltam") e corresponde agora à mesma sequência de paragens que o simulador Python usa para interpolar o GPS do autocarro. Corre esta célula uma vez para (re)criar `route_stops` sempre que o catálogo for reiniciado ou limpo — antes de correr a Notebook 1.1.


In [0]:
from pyspark.sql import Row

# Substitui a tabela route_stops anterior (72 paragens, incompleta para a Linha 202 —
# faltavam as 14 primeiras paragens, Av.Aliados -> Av. do Bessa).
# Fonte: API publica stcp.pt/api/route/<linha>/stops/direction, consultada em 2026-10-15.
# Estas 86 paragens sao as mesmas usadas pelo simulador Python para interpolar o
# GPS real do autocarro (ver ROUTE_STOPS em electric_bus_simulator.py).
paragens = [
    # LINHA 200 - BUS-001 - BOLHÃO -> CASTELO DO QUEIJO
    Row(bus_id="BUS-001", route_id="200", stop_sequence=1, stop_id="BLRB1", stop_name='Bolhão', arrival_time="06:00", distance_meters=0.0, stop_lat=41.151966, stop_lon=-8.607095),
    Row(bus_id="BUS-001", route_id="200", stop_sequence=2, stop_id="MCBL", stop_name='Mercado do Bolhão', arrival_time="06:00", distance_meters=275.53, stop_lat=41.149509, stop_lon=-8.607564),
    Row(bus_id="BUS-001", route_id="200", stop_sequence=3, stop_id="PRDJ", stop_name='Pr. D. João I', arrival_time="06:01", distance_meters=526.45, stop_lat=41.147578, stop_lon=-8.609125),
    Row(bus_id="BUS-001", route_id="200", stop_sequence=4, stop_id="PRFL", stop_name='Pr. Filipa Lencastre', arrival_time="06:02", distance_meters=843.45, stop_lat=41.148299, stop_lon=-8.612795),
    Row(bus_id="BUS-001", route_id="200", stop_sequence=5, stop_id="GGF", stop_name='Guil. G. Fernandes', arrival_time="06:02", distance_meters=1027.57, stop_lat=41.147560, stop_lon=-8.614767),
    Row(bus_id="BUS-001", route_id="200", stop_sequence=6, stop_id="CMO", stop_name='Carmo', arrival_time="06:03", distance_meters=1210.61, stop_lat=41.147221, stop_lon=-8.616910),
    Row(bus_id="BUS-001", route_id="200", stop_sequence=7, stop_id="HSA5", stop_name='Hosp. St. António', arrival_time="06:04", distance_meters=1677.6, stop_lat=41.147766, stop_lon=-8.622450),
    Row(bus_id="BUS-001", route_id="200", stop_sequence=8, stop_id="PAL2", stop_name='Palácio', arrival_time="06:05", distance_meters=1971.17, stop_lat=41.149145, stop_lon=-8.625447),
    Row(bus_id="BUS-001", route_id="200", stop_sequence=9, stop_id="PRG4", stop_name='Pr. da Galiza', arrival_time="06:06", distance_meters=2369.93, stop_lat=41.152531, stop_lon=-8.627041),
    Row(bus_id="BUS-001", route_id="200", stop_sequence=10, stop_id="JM1", stop_name='Junta Massarelos', arrival_time="06:07", distance_meters=2719.76, stop_lat=41.152694, stop_lon=-8.631221),
    Row(bus_id="BUS-001", route_id="200", stop_sequence=11, stop_id="GGT1", stop_name='Gólgota', arrival_time="06:08", distance_meters=2926.54, stop_lat=41.152639, stop_lon=-8.633694),
    Row(bus_id="BUS-001", route_id="200", stop_sequence=12, stop_id="PLNT1", stop_name='Planetário', arrival_time="06:09", distance_meters=3313.89, stop_lat=41.153094, stop_lon=-8.638289),
    Row(bus_id="BUS-001", route_id="200", stop_sequence=13, stop_id="FCUP1", stop_name='Faculdade de Ciências', arrival_time="06:09", distance_meters=3579.46, stop_lat=41.153838, stop_lon=-8.641309),
    Row(bus_id="BUS-001", route_id="200", stop_sequence=14, stop_id="JB1", stop_name='Jardim Botânico', arrival_time="06:10", distance_meters=3834.68, stop_lat=41.154528, stop_lon=-8.644222),
    Row(bus_id="BUS-001", route_id="200", stop_sequence=15, stop_id="LRD1", stop_name='Lordelo', arrival_time="06:11", distance_meters=4236.64, stop_lat=41.154389, stop_lon=-8.649028),
    Row(bus_id="BUS-001", route_id="200", stop_sequence=16, stop_id="PLM1", stop_name='Palmeiras', arrival_time="06:12", distance_meters=4578.91, stop_lat=41.152356, stop_lon=-8.652107),
    Row(bus_id="BUS-001", route_id="200", stop_sequence=17, stop_id="FLUN1", stop_name='Fluvial (Norte)', arrival_time="06:13", distance_meters=4838.04, stop_lat=41.152139, stop_lon=-8.655194),
    Row(bus_id="BUS-001", route_id="200", stop_sequence=18, stop_id="PG1", stop_name='Paulo da Gama', arrival_time="06:14", distance_meters=5164.36, stop_lat=41.150856, stop_lon=-8.658707),
    Row(bus_id="BUS-001", route_id="200", stop_sequence=19, stop_id="PT3", stop_name='Pasteleira', arrival_time="06:15", distance_meters=5400.4, stop_lat=41.150486, stop_lon=-8.661488),
    Row(bus_id="BUS-001", route_id="200", stop_sequence=20, stop_id="TRR3", stop_name='Torres', arrival_time="06:15", distance_meters=5663.72, stop_lat=41.150749, stop_lon=-8.664619),
    Row(bus_id="BUS-001", route_id="200", stop_sequence=21, stop_id="PCL1", stop_name='Padre Luis Cabral', arrival_time="06:16", distance_meters=5974.02, stop_lat=41.151806, stop_lon=-8.668056),
    Row(bus_id="BUS-001", route_id="200", stop_sequence=22, stop_id="UC1", stop_name='Univ. Católica', arrival_time="06:17", distance_meters=6280.78, stop_lat=41.153538, stop_lon=-8.670916),
    Row(bus_id="BUS-001", route_id="200", stop_sequence=23, stop_id="MFZ1", stop_name='Mercado da Foz', arrival_time="06:18", distance_meters=6543.91, stop_lat=41.155048, stop_lon=-8.673343),
    Row(bus_id="BUS-001", route_id="200", stop_sequence=24, stop_id="LIEG1", stop_name='Pr. de Liége', arrival_time="06:19", distance_meters=6863.62, stop_lat=41.155089, stop_lon=-8.677168),
    Row(bus_id="BUS-001", route_id="200", stop_sequence=25, stop_id="CRTO3", stop_name='Crasto', arrival_time="06:20", distance_meters=7226.91, stop_lat=41.157713, stop_lon=-8.679766),
    Row(bus_id="BUS-001", route_id="200", stop_sequence=26, stop_id="MLH5", stop_name='Molhe', arrival_time="06:20", distance_meters=7514.85, stop_lat=41.159880, stop_lon=-8.681660),
    Row(bus_id="BUS-001", route_id="200", stop_sequence=27, stop_id="FCH1", stop_name='Funchal', arrival_time="06:21", distance_meters=7912.82, stop_lat=41.163190, stop_lon=-8.683490),
    Row(bus_id="BUS-001", route_id="200", stop_sequence=28, stop_id="HMLM3", stop_name='Homem do Leme', arrival_time="06:22", distance_meters=8121.71, stop_lat=41.163454, stop_lon=-8.685965),
    Row(bus_id="BUS-001", route_id="200", stop_sequence=29, stop_id="TIM1", stop_name='Timor', arrival_time="06:23", distance_meters=8379.0, stop_lat=41.165587, stop_lon=-8.687170),
    Row(bus_id="BUS-001", route_id="200", stop_sequence=30, stop_id="CQ10", stop_name='Castelo do Queijo', arrival_time="06:24", distance_meters=8640.21, stop_lat=41.167422, stop_lon=-8.689127),

    # LINHA 201 - BUS-002 - AV. ALIADOS -> VISO
    Row(bus_id="BUS-002", route_id="201", stop_sequence=1, stop_id="AL1", stop_name='Av. Aliados', arrival_time="07:38", distance_meters=0.0, stop_lat=41.147208, stop_lon=-8.610983),
    Row(bus_id="BUS-002", route_id="201", stop_sequence=2, stop_id="TRD6", stop_name='Trindade', arrival_time="07:39", distance_meters=424.71, stop_lat=41.151033, stop_lon=-8.610854),
    Row(bus_id="BUS-002", route_id="201", stop_sequence=3, stop_id="PRFL", stop_name='Pr. Filipa Lencastre', arrival_time="07:40", distance_meters=768.79, stop_lat=41.148299, stop_lon=-8.612795),
    Row(bus_id="BUS-002", route_id="201", stop_sequence=4, stop_id="GGF", stop_name='Guil. G. Fernandes', arrival_time="07:40", distance_meters=952.92, stop_lat=41.147560, stop_lon=-8.614767),
    Row(bus_id="BUS-002", route_id="201", stop_sequence=5, stop_id="CMO", stop_name='Carmo', arrival_time="07:41", distance_meters=1135.96, stop_lat=41.147221, stop_lon=-8.616910),
    Row(bus_id="BUS-002", route_id="201", stop_sequence=6, stop_id="HSA5", stop_name='Hosp. St. António', arrival_time="07:42", distance_meters=1602.95, stop_lat=41.147766, stop_lon=-8.622450),
    Row(bus_id="BUS-002", route_id="201", stop_sequence=7, stop_id="PAL3", stop_name='Palácio', arrival_time="07:43", distance_meters=1925.32, stop_lat=41.149512, stop_lon=-8.625532),
    Row(bus_id="BUS-002", route_id="201", stop_sequence=8, stop_id="PRG1", stop_name='Pr. da Galiza', arrival_time="07:44", distance_meters=2366.88, stop_lat=41.153444, stop_lon=-8.626333),
    Row(bus_id="BUS-002", route_id="201", stop_sequence=9, stop_id="BS1", stop_name='Boavista - B.Sucesso', arrival_time="07:45", distance_meters=2721.41, stop_lat=41.156403, stop_lon=-8.627930),
    Row(bus_id="BUS-002", route_id="201", stop_sequence=10, stop_id="BCM1", stop_name='Boavista-Casa da Música', arrival_time="07:46", distance_meters=3030.34, stop_lat=41.158950, stop_lon=-8.629420),
    Row(bus_id="BUS-002", route_id="201", stop_sequence=11, stop_id="AGM1", stop_name='Agramonte', arrival_time="07:47", distance_meters=3462.81, stop_lat=41.158948, stop_lon=-8.634595),
    Row(bus_id="BUS-002", route_id="201", stop_sequence=12, stop_id="ACRD1", stop_name='António Cardoso', arrival_time="07:48", distance_meters=3918.62, stop_lat=41.159843, stop_lon=-8.639918),
    Row(bus_id="BUS-002", route_id="201", stop_sequence=13, stop_id="BSS1", stop_name='Bessa', arrival_time="07:49", distance_meters=4272.82, stop_lat=41.160520, stop_lon=-8.644060),
    Row(bus_id="BUS-002", route_id="201", stop_sequence=14, stop_id="FOCO1", stop_name='Foco', arrival_time="07:50", distance_meters=4563.92, stop_lat=41.161090, stop_lon=-8.647460),
    Row(bus_id="BUS-002", route_id="201", stop_sequence=15, stop_id="PINM1", stop_name='Pinheiro Manso', arrival_time="07:51", distance_meters=4946.5, stop_lat=41.161845, stop_lon=-8.651927),
    Row(bus_id="BUS-002", route_id="201", stop_sequence=16, stop_id="SRV3", stop_name='Serralves', arrival_time="07:53", distance_meters=5462.32, stop_lat=41.162848, stop_lon=-8.657954),
    Row(bus_id="BUS-002", route_id="201", stop_sequence=17, stop_id="PNV2", stop_name='Paulo Novais', arrival_time="07:53", distance_meters=5642.35, stop_lat=41.163200, stop_lon=-8.660057),
    Row(bus_id="BUS-002", route_id="201", stop_sequence=18, stop_id="FTM1", stop_name='Fonte da Moura', arrival_time="07:54", distance_meters=5869.06, stop_lat=41.164505, stop_lon=-8.662144),
    Row(bus_id="BUS-002", route_id="201", stop_sequence=19, stop_id="BRV1", stop_name='Bairro Vilarinha', arrival_time="07:55", distance_meters=6205.43, stop_lat=41.167065, stop_lon=-8.659990),
    Row(bus_id="BUS-002", route_id="201", stop_sequence=20, stop_id="LDD1", stop_name='Lidador', arrival_time="07:56", distance_meters=6564.91, stop_lat=41.169798, stop_lon=-8.657682),
    Row(bus_id="BUS-002", route_id="201", stop_sequence=21, stop_id="PRO1", stop_name='Pereiró', arrival_time="07:57", distance_meters=6903.6, stop_lat=41.172368, stop_lon=-8.655497),
    Row(bus_id="BUS-002", route_id="201", stop_sequence=22, stop_id="EZC1", stop_name='Ezequiel Campos', arrival_time="07:57", distance_meters=7174.94, stop_lat=41.174449, stop_lon=-8.653793),
    Row(bus_id="BUS-002", route_id="201", stop_sequence=23, stop_id="MPAZ5", stop_name='Man.P.Azevedo 3', arrival_time="07:58", distance_meters=7500.86, stop_lat=41.176989, stop_lon=-8.651836),
    Row(bus_id="BUS-002", route_id="201", stop_sequence=24, stop_id="IMTT1", stop_name='IMTT', arrival_time="07:59", distance_meters=7803.96, stop_lat=41.178485, stop_lon=-8.648801),
    Row(bus_id="BUS-002", route_id="201", stop_sequence=25, stop_id="VIS5", stop_name='Viso (Metro)', arrival_time="08:00", distance_meters=8033.14, stop_lat=41.177361, stop_lon=-8.646500),
    Row(bus_id="BUS-002", route_id="201", stop_sequence=26, stop_id="VIS3", stop_name='Viso', arrival_time="08:01", distance_meters=8389.7, stop_lat=41.177840, stop_lon=-8.642280),

    # LINHA 202 - BUS-003 - AV.ALIADOS -> PASSEIO ALEGRE
    Row(bus_id="BUS-003", route_id="202", stop_sequence=1, stop_id="AAL4", stop_name='Av.Aliados', arrival_time="06:43", distance_meters=0.0, stop_lat=41.148528, stop_lon=-8.610667),
    Row(bus_id="BUS-003", route_id="202", stop_sequence=2, stop_id="TRD2", stop_name='Trindade', arrival_time="06:44", distance_meters=362.94, stop_lat=41.151695, stop_lon=-8.609587),
    Row(bus_id="BUS-003", route_id="202", stop_sequence=3, stop_id="GCRT1", stop_name='Gonçalo Cristóvão', arrival_time="06:44", distance_meters=640.34, stop_lat=41.154194, stop_lon=-8.609611),
    Row(bus_id="BUS-003", route_id="202", stop_sequence=4, stop_id="PRR1", stop_name='Pr. da República', arrival_time="06:45", distance_meters=834.0, stop_lat=41.154023, stop_lon=-8.611917),
    Row(bus_id="BUS-003", route_id="202", stop_sequence=5, stop_id="FIG", stop_name='Figueiroa', arrival_time="06:46", distance_meters=1299.22, stop_lat=41.155956, stop_lon=-8.616856),
    Row(bus_id="BUS-003", route_id="202", stop_sequence=6, stop_id="CVA1", stop_name='Carvalhosa', arrival_time="06:47", distance_meters=1582.57, stop_lat=41.156512, stop_lon=-8.620165),
    Row(bus_id="BUS-003", route_id="202", stop_sequence=7, stop_id="HML1", stop_name='Hospital Militar', arrival_time="06:48", distance_meters=1929.22, stop_lat=41.157176, stop_lon=-8.624218),
    Row(bus_id="BUS-003", route_id="202", stop_sequence=8, stop_id="BVBR2", stop_name='Boavista Brasília', arrival_time="06:48", distance_meters=2139.32, stop_lat=41.157587, stop_lon=-8.626672),
    Row(bus_id="BUS-003", route_id="202", stop_sequence=9, stop_id="BCM3", stop_name='Boavista-Casa da Música', arrival_time="06:49", distance_meters=2403.15, stop_lat=41.159458, stop_lon=-8.628619),
    Row(bus_id="BUS-003", route_id="202", stop_sequence=10, stop_id="GRC1", stop_name='Graciosa', arrival_time="06:50", distance_meters=2798.71, stop_lat=41.162921, stop_lon=-8.627502),
    Row(bus_id="BUS-003", route_id="202", stop_sequence=11, stop_id="CSBV", stop_name='Casa de Saúde da Boavista', arrival_time="06:51", distance_meters=3225.4, stop_lat=41.164532, stop_lon=-8.632138),
    Row(bus_id="BUS-003", route_id="202", stop_sequence=12, stop_id="SDP1", stop_name='Sidónio Pais', arrival_time="06:53", distance_meters=3632.04, stop_lat=41.164563, stop_lon=-8.637004),
    Row(bus_id="BUS-003", route_id="202", stop_sequence=13, stop_id="ABSS1", stop_name='Av. do Bessa', arrival_time="06:54", distance_meters=4147.63, stop_lat=41.165323, stop_lon=-8.643091),
    Row(bus_id="BUS-003", route_id="202", stop_sequence=14, stop_id="SJBC1", stop_name='S.João Bosco', arrival_time="06:55", distance_meters=4554.56, stop_lat=41.166131, stop_lon=-8.647841),
    Row(bus_id="BUS-003", route_id="202", stop_sequence=15, stop_id="SJB1", stop_name='S. João de Brito', arrival_time="06:56", distance_meters=4787.37, stop_lat=41.166494, stop_lon=-8.650585),
    Row(bus_id="BUS-003", route_id="202", stop_sequence=16, stop_id="ABM1", stop_name='Alberto Macedo', arrival_time="06:57", distance_meters=5121.31, stop_lat=41.167151, stop_lon=-8.654485),
    Row(bus_id="BUS-003", route_id="202", stop_sequence=17, stop_id="BRP1", stop_name='Bairro da Previdência', arrival_time="06:58", distance_meters=5426.56, stop_lat=41.165783, stop_lon=-8.657654),
    Row(bus_id="BUS-003", route_id="202", stop_sequence=18, stop_id="BRV3", stop_name='Bairro da Vilarinha', arrival_time="06:58", distance_meters=5576.9, stop_lat=41.166094, stop_lon=-8.659405),
    Row(bus_id="BUS-003", route_id="202", stop_sequence=19, stop_id="FTM2", stop_name='Fonte da Moura', arrival_time="06:59", distance_meters=5920.36, stop_lat=41.164191, stop_lon=-8.662646),
    Row(bus_id="BUS-003", route_id="202", stop_sequence=20, stop_id="ABVT1", stop_name='Av. da Boavista', arrival_time="06:59", distance_meters=6084.85, stop_lat=41.162823, stop_lon=-8.663403),
    Row(bus_id="BUS-003", route_id="202", stop_sequence=21, stop_id="LGO4", stop_name='Liceu Garcia Orta', arrival_time="07:00", distance_meters=6401.41, stop_lat=41.161528, stop_lon=-8.666778),
    Row(bus_id="BUS-003", route_id="202", stop_sequence=22, stop_id="RCRT2", stop_name='R.Crasto', arrival_time="07:01", distance_meters=6815.55, stop_lat=41.162000, stop_lon=-8.671694),
    Row(bus_id="BUS-003", route_id="202", stop_sequence=23, stop_id="LNEV2", stop_name='Lgo.Nevogilde', arrival_time="07:03", distance_meters=7211.9, stop_lat=41.162833, stop_lon=-8.676306),
    Row(bus_id="BUS-003", route_id="202", stop_sequence=24, stop_id="NEVG", stop_name='Nevogilde', arrival_time="07:03", distance_meters=7496.92, stop_lat=41.161511, stop_lon=-8.679230),
    Row(bus_id="BUS-003", route_id="202", stop_sequence=25, stop_id="MLH3", stop_name='Molhe', arrival_time="07:04", distance_meters=7859.95, stop_lat=41.160158, stop_lon=-8.683185),
    Row(bus_id="BUS-003", route_id="202", stop_sequence=26, stop_id="JNN2", stop_name='Jacinto Nunes', arrival_time="07:05", distance_meters=8125.03, stop_lat=41.158107, stop_lon=-8.681560),
    Row(bus_id="BUS-003", route_id="202", stop_sequence=27, stop_id="CRTO4", stop_name='Crasto', arrival_time="07:06", distance_meters=8324.5, stop_lat=41.156600, stop_lon=-8.680260),
    Row(bus_id="BUS-003", route_id="202", stop_sequence=28, stop_id="PING3", stop_name='Praia dos Ingleses', arrival_time="07:07", distance_meters=8772.97, stop_lat=41.152767, stop_lon=-8.678563),
    Row(bus_id="BUS-003", route_id="202", stop_sequence=29, stop_id="PORI2", stop_name='Praia do Ourigo', arrival_time="07:08", distance_meters=9208.48, stop_lat=41.149376, stop_lon=-8.675942),
    Row(bus_id="BUS-003", route_id="202", stop_sequence=30, stop_id="PASS1", stop_name='Passeio Alegre', arrival_time="07:09", distance_meters=9482.32, stop_lat=41.148820, stop_lon=-8.672750),
]

df_paragens = spark.createDataFrame(paragens)
df_paragens.write.format("delta").mode("overwrite").saveAsTable("route_stops")
print(f"Tabela route_stops criada com {df_paragens.count()} paragens")


Tabela route_stops criada com 86 paragens


## Passo 8: Ciclo de Carregamento dos Autocarros

O simulador Python tem um ciclo fechado: um autocarro
que atinge o SOC mínimo deixa de terminar a simulação — segue para a Central de
Carregamento (Via Norte / Bonjóia), aguarda uma estação livre, carrega até 80% e
regressa à sua rota. Estas células mostram esse processo a partir da telemetria real.


Como `bus_telemetry` e `charger_telemetry` guardam todos os eventos recebidos nos últimos 10 minutos (não apenas o mais recente por dispositivo), as três queries seguintes aplicam `ROW_NUMBER() OVER (PARTITION BY device_id ORDER BY event_time DESC)` para isolar o estado mais atual de cada autocarro e estação antes de o apresentar.

In [0]:
print("=== CICLO DE CARREGAMENTO - ESTADO ATUAL (autocarro x estacao) ===")

display(spark.sql("""
    WITH latest_bus AS (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY device_id ORDER BY event_time DESC) as rn
        FROM bus_telemetry
    ),
    latest_charger AS (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY device_id ORDER BY event_time DESC) as rn
        FROM charger_telemetry
    )
    SELECT
        b.device_id AS autocarro,
        b.state AS estado_autocarro,
        ROUND(b.batteryLevel, 1) AS bateria_pct,
        b.currentStop AS paragem_atual,
        c.device_id AS estacao,
        c.state AS estado_estacao,
        ROUND(c.currentPower, 1) AS potencia_kw,
        ROUND(c.energyDelivered, 2) AS energia_entregue_kwh
    FROM latest_bus b
    LEFT JOIN latest_charger c
        ON c.connectedBus = b.device_id AND c.rn = 1
    WHERE b.rn = 1
    ORDER BY b.device_id
"""))


=== CICLO DE CARREGAMENTO - ESTADO ATUAL (autocarro x estacao) ===


autocarro,estado_autocarro,bateria_pct,paragem_atual,estacao,estado_estacao,potencia_kw,energia_entregue_kwh
BUS-001,parked,70.2,Castelo do Queijo,null,null,null,null
BUS-002,waitingForCharger,18.3,Central de Carregamento,null,null,null,null
BUS-003,charging,27.3,Central de Carregamento,CS-003,occupied,120.0,16.5


In [0]:
print("=== AUTOCARROS A CAMINHO DA CENTRAL OU EM FILA DE ESPERA ===")

display(spark.sql("""
    SELECT
        device_id,
        state,
        ROUND(batteryLevel, 1) as bateria_pct,
        ROUND(latitude, 6) as latitude,
        ROUND(longitude, 6) as longitude
    FROM (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY device_id ORDER BY event_time DESC) as rn
        FROM bus_telemetry
    )
    WHERE rn = 1 AND state IN ('headingToCharging', 'waitingForCharger')
"""))


=== AUTOCARROS A CAMINHO DA CENTRAL OU EM FILA DE ESPERA ===


device_id,state,bateria_pct,latitude,longitude
BUS-002,waitingForCharger,18.3,41.152661,-8.579658


In [0]:
print("=== ESTADO ATUAL DAS ESTACOES DE CARREGAMENTO ===")

display(spark.sql("""
    SELECT
        device_id,
        state,
        connectedBus,
        ROUND(currentPower, 1) as potencia_kw,
        ROUND(energyDelivered, 2) as energia_entregue_kwh,
        ROUND(chargerTemperature, 1) as temperatura_C
    FROM (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY device_id ORDER BY event_time DESC) as rn
        FROM charger_telemetry
    )
    WHERE rn = 1
    ORDER BY device_id
"""))


=== ESTADO ATUAL DAS ESTACOES DE CARREGAMENTO ===


device_id,state,connectedBus,potencia_kw,energia_entregue_kwh,temperatura_C
CS-001,occupied,BUS-004,80.0,30.61,45.0
CS-002,fault,null,0.0,0.0,28.4
CS-003,occupied,BUS-003,120.0,17.0,31.9


## Passo 9: Latência de ingestão (Event Hub → Delta)

Mede o intervalo entre o instante em que cada evento chega ao Event Hub (`enqueued_time`) e o instante em que é processado e escrito na tabela Delta (`event_time`), em segundos. Esta latência fim a fim é o indicador direto usado para avaliar o requisito de processamento em tempo quase real (RNF01).

In [0]:
%sql
SELECT
    COUNT(*) AS n_eventos,
    ROUND(AVG(unix_timestamp(event_time) - unix_timestamp(enqueued_time)), 3) AS latencia_media_seg,
    ROUND(MIN(unix_timestamp(event_time) - unix_timestamp(enqueued_time)), 3) AS latencia_min_seg,
    ROUND(MAX(unix_timestamp(event_time) - unix_timestamp(enqueued_time)), 3) AS latencia_max_seg,
    ROUND(PERCENTILE(unix_timestamp(event_time) - unix_timestamp(enqueued_time), 0.5), 3) AS latencia_mediana_seg,
    ROUND(PERCENTILE(unix_timestamp(event_time) - unix_timestamp(enqueued_time), 0.95), 3) AS latencia_p95_seg
FROM bus_telemetry_raw

n_eventos,latencia_media_seg,latencia_min_seg,latencia_max_seg,latencia_mediana_seg,latencia_p95_seg
600,2.22,0,29,2.0,5.0
